In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Let A be the first node and B be the last.
Nodes = ['A','X1','X2','X3','X4','X5','X6','Y1','Y2','B']

Links = [['A','X1',10,0.01],
         ['A','X2',10,0.01],
         ['A','Y1',150,0.01],
         ['X1','X2',10,0.01],
         ['X1','X6',80,0.001],
         ['X1','X3',10,0.01],
         ['X2','X3',1,0.001],
         ['X2','X4',10,0.01],
         ['X2','Y1',100,0.01],
         ['X2','X5',80,0.001],
         ['X3','X5',10,0.01],
         ['X3','X4',10,0.01],
         ['X4','X5',1,0.001],
         ['X4','X6',40,0.001],
         ['X4','Y1',2,0.001],
         ['X5','X6',10,0.01],
         ['X5','B',10,0.01],
         ['X6','B',10,0.01],
         ['Y1','Y2',100,0.01],
         ['Y2','B',150,0.01]]

In [ ]:
Total_drivers = 34000
L_rate = 0.00001

def equil(Paths,Link_dict):
  iterations = 5000

  Node_path = {}

  for path_idx, path in enumerate(Paths):

      for i in range(len(path) - 1):
        source = path[i]
        target = path[i + 1]
        link = (source, target)

        if link not in Node_path:
            Node_path[link] = []

        Node_path[link].append(path_idx)

  Path_driver = np.ones(len(Paths))*Total_drivers/len(Paths)
  cost = np.zeros(len(Paths))
  cost_upd = np.zeros(len(Paths))
  cost_diff = np.zeros(len(Paths))

  for i in range(iterations):
    for j in range(len(Paths)):
      cost[j] = get_cost(Path_driver,Paths[j],Node_path,Link_dict)
    avg_cost = cost.sum()/len(cost)

    frac_pathdri = np.zeros(len(Paths))
    del_frac = np.zeros(len(Paths))
    frac_pathdri_upd = np.zeros(len(Paths))

    frac_pathdri = Path_driver/Total_drivers
    for j in range(len(Paths)):
      del_frac[j] = L_rate*frac_pathdri[j]*(avg_cost-cost[j])

    frac_pathdri_upd = frac_pathdri + del_frac

    frac_pathdri_upd = np.clip(frac_pathdri_upd, 0, 1)

    if frac_pathdri_upd.sum() == 0:
      frac_pathdri_upd = np.ones(len(Paths)) / len(Paths)

    frac_pathdri_upd = frac_pathdri_upd / frac_pathdri_upd.sum()

    Path_driver = frac_pathdri_upd * Total_drivers
    #Path_driver[-1] = Total_drivers - Path_driver[:-1].sum()

    for j in range(len(Paths)):
      cost_upd[j] = get_cost(Path_driver,Paths[j],Node_path,Link_dict)

    cost_diff = abs(cost - cost_upd)
    if cost_diff.max() < 0.01:
      print(f"Convergence achieved at {i+1}th iteration")
      break

  return cost_upd.sum()/len(cost_upd),Path_driver

def get_cost(Path_driver,Path,Node_path,Link_dict):
  cost = 0
  for i in range(len(Path)-1):
    S = Path[i]
    T = Path[i+1]
    link = (S,T)

    Path_with_link = Node_path[link]
    for p in Path_with_link:
      cost += Link_dict[link][0] + Link_dict[link][1]*Path_driver[p]

  return cost


In [ ]:
def equilibrium(Nodes,Links):
  Link_dict = {(source, target): [cost, prob] for source, target, cost, prob in Links}

  node_dict = {node: 0 for node in Nodes}

  adj_list = {}
  for source,dest,base_cost,conj_cost in Links:
    if source not in adj_list:
        adj_list[source] = []
    adj_list[source].append(dest)

  Paths = []

  current_path = ['A']
  current_node = 'A'


  while len(current_path) > 0:

    # Current node has exhausted all its connected neighbors
    if current_node not in adj_list or node_dict[current_node] == len(adj_list[current_node]):
        if current_node in node_dict:
            node_dict[current_node] = 0  #Reseting neighbor counter for future paths

        current_path.pop()

        if len(current_path) == 0:
            break  # Tree fully explored
        current_node = current_path[-1]

    else:
        # Get the next target node using your pointer tracker
        neighbor_node = adj_list[current_node][node_dict[current_node]]

        # The next connected node is the destination 'B'
        if neighbor_node == 'B':
            current_path.append('B')
            Paths.append(list(current_path))
            node_dict[current_node] += 1
            current_path.pop()

        # The next node is a regular node
        elif neighbor_node != 'B':
            if neighbor_node not in current_path:
                node_dict[current_node] += 1  # Advance pointer on current node
                current_path.append(neighbor_node)  # Push to path
                current_node = neighbor_node  # Move position to the next node
            else:
                # If already in path, skip it
                node_dict[current_node] += 1



  time,drivers = equil(Paths,Link_dict)

  return time,drivers,Paths

In [ ]:
time,drivers,Paths = equilibrium(Nodes,Links)
print("The time taken is",time,"units")

formatted_drivers = [f"{driver:.2f}" for driver in drivers]
print("The number of drivers in each path are:", formatted_drivers)

Convergence achieved at 3255th iteration
The time taken is 1338.1809026966887 units
The number of drivers in each path are: ['0.13', '16.01', '0.04', '4.48', '0.01', '0.00', '0.11', '13.99', '0.02', '0.00', '0.00', '0.00', '0.13', '15421.71', '2.40', '303.41', '0.67', '85.05', '0.12', '0.00', '66.03', '8320.60', '18.50', '2335.97', '3.35', '0.00', '57.67', '7269.20', '10.44', '0.00', '0.00', '0.55', '69.42', '0.00']


In [ ]:
Times = np.zeros(len(Links))
for i in range(len(Links)):
  Links_new = Links.copy()
  Links_new.remove(Links[i])
  time,drivers,Paths = equilibrium(Nodes,Links_new)
  Times[i] = time

print("The minimum time taken is",Times.min(),"units when the link removed is",Links[Times.argmin()])
print("The maximum time taken is",Times.max(),"units when the link removed is",Links[Times.argmax()])

Convergence achieved at 2023th iteration
Convergence achieved at 2642th iteration
Convergence achieved at 3252th iteration
Convergence achieved at 3693th iteration
Convergence achieved at 3121th iteration
Convergence achieved at 2874th iteration
Convergence achieved at 3463th iteration
Convergence achieved at 2955th iteration
Convergence achieved at 3251th iteration
Convergence achieved at 3349th iteration
Convergence achieved at 2679th iteration
Convergence achieved at 3439th iteration
Convergence achieved at 2755th iteration
Convergence achieved at 3348th iteration
Convergence achieved at 3255th iteration
Convergence achieved at 2828th iteration
Convergence achieved at 3670th iteration
Convergence achieved at 2539th iteration
Convergence achieved at 3298th iteration
Convergence achieved at 3298th iteration
The minimum time taken is 909.3715906512184 units when the link removed is ['Y1', 'Y2', 100, 0.01]
The maximum time taken is 1586.6508931257601 units when the link removed is ['X6'

In [ ]:
for j in range(len(Links)):
  time_0,drivers,Paths = equilibrium(Nodes,Links)
  for i in range(len(Links)):
    Links_new = Links.copy()
    Links_new.remove(Links[i])
    time,drivers,Paths = equilibrium(Nodes,Links_new)
    Times[i] = time

  if Times.min() < time_0:
    print("The link removed is",Links[Times.argmin()])
    Links.remove(Links[Times.argmin()])
  else:
    break

print("The final Links are :",Links)

Convergence achieved at 3255th iteration
Convergence achieved at 2023th iteration
Convergence achieved at 2642th iteration
Convergence achieved at 3252th iteration
Convergence achieved at 3693th iteration
Convergence achieved at 3121th iteration
Convergence achieved at 2874th iteration
Convergence achieved at 3463th iteration
Convergence achieved at 2955th iteration
Convergence achieved at 3251th iteration
Convergence achieved at 3349th iteration
Convergence achieved at 2679th iteration
Convergence achieved at 3439th iteration
Convergence achieved at 2755th iteration
Convergence achieved at 3348th iteration
Convergence achieved at 3255th iteration
Convergence achieved at 2828th iteration
Convergence achieved at 3670th iteration
Convergence achieved at 2539th iteration
Convergence achieved at 3298th iteration
Convergence achieved at 3298th iteration
The link removed is ['Y1', 'Y2', 100, 0.01]
Convergence achieved at 3298th iteration
Convergence achieved at 1972th iteration
Convergence a

In [ ]:
time,drivers,Paths = equilibrium(Nodes,Links)
print("The time taken is",time,"units")
print("The Paths are",Paths)
formatted_drivers = [f"{driver:.2f}" for driver in drivers]
print("The number of drivers in each path are:", formatted_drivers)

Convergence achieved at 1th iteration
The time taken is 457.0 units
The Paths are [['A', 'X1', 'X6', 'B'], ['A', 'X2', 'X5', 'B']]
The number of drivers in each path are: ['17000.00', '17000.00']
